# Modelos no lineales para Regresión

En esta clase estudiaremos diferentes enfoques de **modelos no lineales de regresión**, los cuales permiten capturar relaciones más complejas entre las variables independientes (X) y la variable dependiente (Y), superando las limitaciones de los modelos lineales tradicionales.

---

## Capítulo 1: ¿Qué otros modelos existen para Regresión?

Además de la regresión lineal, existen diversos modelos no lineales que pueden aplicarse para predecir variables continuas:

- 🔹 **K-Nearest Neighbors (KNN)**: Basado en la similitud entre observaciones. La predicción se realiza tomando el promedio de los vecinos más cercanos al punto de interés.  
- 🔹 **Support Vector Regression (SVR)**: Extensión de las máquinas de soporte vectorial al caso de regresión. Busca un hiperplano que aproxime los datos dentro de un margen de tolerancia.  
- 🔹 **Random Forest Regressor**: Combina múltiples árboles de decisión mediante ensamble (bagging) para mejorar precisión y reducir el sobreajuste.   

Estos modelos permiten aproximar relaciones no lineales y suelen tener mejor desempeño en problemas con datos de alta complejidad.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/housing/housing.data',header=None,sep=r'\s+')
df.columns =  ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV']
df.head(5)

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2


In [3]:
corr_matrix = df.corr()
corr_medv = corr_matrix[['MEDV']].sort_values(by='MEDV',ascending=False)
corr_medv

,MEDV
MEDV,1.000000
RM,0.695360
ZN,0.360445
B,0.333461
DIS,0.249929
CHAS,0.175260
AGE,-0.376955
RAD,-0.381626
CRIM,-0.388305
NOX,-0.427321


In [4]:
cols = corr_medv.index.tolist()
cols.remove('MEDV')
cols

['RM',
 'ZN',
 'B',
 'DIS',
 'CHAS',
 'AGE',
 'RAD',
 'CRIM',
 'NOX',
 'TAX',
 'INDUS',
 'PTRATIO',
 'LSTAT']

In [5]:
X = df[cols].values
y = df['MEDV'].values.reshape(-1,1)

In [6]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.30)

In [7]:
from sklearn.preprocessing import StandardScaler
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled = scaler_y.transform(y_test)

# CREANDO MODELO KNN

## CREO EL MODELO Y LO ENTRENO

In [8]:
from sklearn.neighbors import KNeighborsRegressor

#creamos el modelo
knn_regressor = KNeighborsRegressor(n_neighbors=5)

#entrenamos el modelo
knn_regressor.fit(X_train_scaled,y_train_scaled)

KNeighborsRegressor()

## EVALUO EL MODELO

In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# Realizar predicciones en el conjunto de prueba
y_pred = knn_regressor.predict(X_test_scaled)

# Calcular las métricas de evaluación
mae = mean_absolute_error(y_test_scaled, y_pred)
mse = mean_squared_error(y_test_scaled, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_scaled, y_pred)

# Mostrar las métricas
print(f'Mean Absolute Error (MAE): {mae}')
print(f'Mean Squared Error (MSE): {mse}')
print(f'Root Mean Squared Error (RMSE): {rmse}')
print(f'R-squared (R2): {r2}')

Mean Absolute Error (MAE): 0.3249331814364445
Mean Squared Error (MSE): 0.2422151946463501
Root Mean Squared Error (RMSE): 0.49215362911021
R-squared (R2): 0.7614681661744154


# ENTRENAMIENTO CON MODELO SRV

## IMPORTAMOS LIBRERIAS

In [11]:
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## CREAMOS MODELO Y ENTRENAMOS

### PARAMETROS
### KERNEL  'LINEAR' 'POLY' 'RBF'
### C = COST
Es el parámetro de regularización (también llamado “cost”).
Controla cuánto penaliza el modelo los errores.

C pequeño (0.1, 1) → modelo más suave, más regularización, menos overfitting

C grande (100, 1000) → modelo se ajusta más a los datos (riesgo de overfitting)

Valor por defecto: C=1.0.
### EPSILON
Define la zona de tolerancia donde los errores no se penalizan.

Ejemplos:

epsilon pequeño (0.01) → el modelo intenta ajustarse más

epsilon grande (0.5, 1) → permite más errores sin penalizar → modelo más suave

Valor por defecto: epsilon=0.1.

In [12]:
svr_regressor = SVR(kernel='rbf',C=10,epsilon=0)
# Entrenar el modelo
svr_regressor.fit(X_train_scaled, y_train_scaled.ravel())

SVR(C=10, epsilon=0)

## EVALUAMOS EL MODELO

In [13]:
# Realizar predicciones en el conjunto de prueba escalado
y_pred_scaled = svr_regressor.predict(X_test_scaled)

# Calcular las métricas de evaluación
mae = mean_absolute_error(y_test_scaled, y_pred_scaled)
mse = mean_squared_error(y_test_scaled, y_pred_scaled)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_scaled, y_pred_scaled)

# Mostrar las métricas
print("Métricas para SVR:")
print(f'Mean Absolute Error (MAE): {mae}')
print(f'Mean Squared Error (MSE): {mse}')
print(f'Root Mean Squared Error (RMSE): {rmse}')
print(f'R-squared (R2): {r2}')

Métricas para SVR:
Mean Absolute Error (MAE): 0.26156708139538465
Mean Squared Error (MSE): 0.13648884477309187
Root Mean Squared Error (RMSE): 0.369443967027602
R-squared (R2): 0.8655867379088401


## ENTRENAMIENTO CON RANDOM FOREST REGRESSOR

## PARAMETROS
### ✔️ `n_estimators=100`

Es el **número de árboles** que tendrá el bosque.

📌 **Más árboles → mejor rendimiento**, pero más costo computacional.

**Valores típicos:**

- `10` → rápido pero menos preciso  
- `100` → equilibrio (valor común)  
- `200`, `300` → más precisión  
- `1000` → modelos muy robustos para datasets grandes  

👉 **Mientras más árboles, menor varianza.**

---

### ✔️ `random_state=42`

Es una **semilla** para asegurar que los resultados sean **reproducibles**.

¿Por qué?

Porque *Random Forest* hace cosas aleatorias:

- elige muestras aleatorias (*bootstrap*)  
- elige subconjuntos aleatorios de variables  

Al poner `random_state=42`:

- garantizamos que cada vez que ejecutes el código, obtengas el **mismo resultado**.

💡 Cualquier número sirve, pero **42** es un estándar en ciencia de datos  
(referencia a *“The Hitchhiker’s Guide to the Galaxy”*).

## ENTRENAMOS EL MODELO

In [14]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Crear el modelo RandomForestRegressor
# Puedes ajustar el número de estimadores (n_estimators) y otros parámetros
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)

# Entrenar el modelo
rf_regressor.fit(X_train, y_train.ravel())

RandomForestRegressor(random_state=42)

## EVALUAMOS EL MODELO

In [15]:
# Realizar predicciones en el conjunto de prueba
y_pred = rf_regressor.predict(X_test)

# Calcular las métricas de evaluación
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

# Mostrar las métricas
print("Métricas para RandomForestRegressor:")
print(f'Mean Absolute Error (MAE): {mae}')
print(f'Mean Squared Error (MSE): {mse}')
print(f'Root Mean Squared Error (RMSE): {rmse}')
print(f'R-squared (R2): {r2}')

Métricas para RandomForestRegressor:
Mean Absolute Error (MAE): 2.415434210526315
Mean Squared Error (MSE): 12.649668157894725
Root Mean Squared Error (RMSE): 3.5566371979574645
R-squared (R2): 0.8514327650654869
